In [1]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage import img_as_float
from sklearn.cluster import KMeans

# 1. Загрузка и предобработка изображения
image = imread('parrots.jpg')
image_float = img_as_float(image)  # Приведение к интервалу [0, 1]

# 2. Создание матрицы объекты-признаки (каждый пиксель - строка с R, G, B)
w, h, d = image_float.shape
pixels = image_float.reshape((w * h, d))

print(f"Размер исходного изображения: {image_float.shape}")
print(f"Размер матрицы признаков: {pixels.shape}")

def calculate_psnr(original, compressed):
    mse = np.mean((original - compressed) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    psnr = 10 * np.log10((max_pixel ** 2) / mse)
    return psnr

# Инициализация переменных для поиска минимального числа кластеров
min_clusters = None

# Перебираем количество кластеров от 1 до 20
for k in range(1, 21):
    # 3. Запуск KMeans
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=241, n_init='auto')
    labels = kmeans.fit_predict(pixels)
    
    # Подготовка массивов для средних и медианных значений
    mean_pixels = np.zeros_like(pixels)
    median_pixels = np.zeros_like(pixels)
    
    # Заполнение кластеров средним и медианным цветом
    for cluster_idx in range(k):
        cluster_mask = (labels == cluster_idx)
        cluster_data = pixels[cluster_mask]
        
        # Среднее значение по кластеру
        mean_pixels[cluster_mask] = np.mean(cluster_data, axis=0)
        # Медианное значение по кластеру
        median_pixels[cluster_mask] = np.median(cluster_data, axis=0)
        
    # Считаем PSNR для обоих вариантов
    psnr_mean = calculate_psnr(pixels, mean_pixels)
    psnr_median = calculate_psnr(pixels, median_pixels)
    
    print(f"Кластеров: {k:2d} | PSNR (Среднее): {psnr_mean:.2f} | PSNR (Медиана): {psnr_median:.2f}")
    
    # 5. Проверка условия (PSNR > 20) для любого из способов
    if (psnr_mean > 20 or psnr_median > 20) and min_clusters is None:
        min_clusters = k

print("\n" + "="*40)
print(f"Ответ (минимальное число кластеров, где PSNR > 20): {min_clusters}")
print("="*40)

Размер исходного изображения: (474, 713, 3)
Размер матрицы признаков: (337962, 3)
Кластеров:  1 | PSNR (Среднее): 9.84 | PSNR (Медиана): 9.46
Кластеров:  2 | PSNR (Среднее): 12.11 | PSNR (Медиана): 11.69
Кластеров:  3 | PSNR (Среднее): 13.17 | PSNR (Медиана): 12.63
Кластеров:  4 | PSNR (Среднее): 14.32 | PSNR (Медиана): 14.00
Кластеров:  5 | PSNR (Среднее): 15.09 | PSNR (Медиана): 14.69
Кластеров:  6 | PSNR (Среднее): 16.57 | PSNR (Медиана): 16.07
Кластеров:  7 | PSNR (Среднее): 17.67 | PSNR (Медиана): 17.37
Кластеров:  8 | PSNR (Среднее): 18.38 | PSNR (Медиана): 18.10
Кластеров:  9 | PSNR (Среднее): 19.14 | PSNR (Медиана): 18.84
Кластеров: 10 | PSNR (Среднее): 19.54 | PSNR (Медиана): 19.23
Кластеров: 11 | PSNR (Среднее): 20.08 | PSNR (Медиана): 19.83
Кластеров: 12 | PSNR (Среднее): 20.58 | PSNR (Медиана): 20.32
Кластеров: 13 | PSNR (Среднее): 21.06 | PSNR (Медиана): 20.83
Кластеров: 14 | PSNR (Среднее): 21.37 | PSNR (Медиана): 21.16
Кластеров: 15 | PSNR (Среднее): 21.65 | PSNR (Медиан